#### Search engine with pre-available langchain tools

In [1]:
# arxiv - research
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

C:\Users\haris\AppData\Local\Temp\ipykernel_19020\61524073.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun


In [2]:
#Used the inbuild tool of wikipedia
api_wiki_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wiki_wrapper)
wiki.name

'wikipedia'

In [3]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv_q_runner = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv_q_runner.name

'arxiv'

In [4]:
tools = [wiki,arxiv_q_runner]

In [5]:
#Custom tools [RAG tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings,ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\HD_Workspace\LangchainProject\QA_Chatbot\My Venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
loader = WebBaseLoader("https://docs.langchain.com/langsmith/evaluate-on-intermediate-steps#1-define-your-llm-pipeline")
docs = loader.load()
docSplitterRule = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectorDb = FAISS.from_documents(docSplitterRule,NVIDIAEmbeddings())
retriver = vectorDb.as_retriever()
retriver

VectorStoreRetriever(tags=['FAISS', 'NVIDIAEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001B16C387B60>, search_kwargs={})

In [7]:
from langchain_core.tools.retriever import create_retriever_tool
retriver_tool = create_retriever_tool(retriver, "langsmith-search", "Seach any informatio about langsmith")
retriver_tool.name

'langsmith-search'

In [8]:
tool=[wiki,arxiv_q_runner,retriver_tool]
tool

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\HD_Workspace\\LangchainProject\\QA_Chatbot\\My Venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Seach any informatio about langsmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001B16C426DE0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001B16CD216C0>)]

In [10]:
# Run all these tools with agent and llm models
# Combine the tools + LLM -> AgentExecution
import os
from dotenv import load_dotenv
load_dotenv()
# os.environ["NVIDIA_API_KEY"]=os.environ("NVIDIA_API_KEY")
nvidia_api_key = os.getenv("NVIDIA_API_KEY")


llm = ChatNVIDIA(nvidia_api_key=nvidia_api_key,model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning")

c:\HD_Workspace\LangchainProject\QA_Chatbot\My Venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found nvidia/nemotron-3-nano-omni-30b-a3b-reasoning in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [ ]:
## Setup complete - Agent is configured
# The agent will be created in the next cell using create_react_agent
pass

ChatPromptTemplate(input_variables=['input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')

In [27]:
# Create Agents using ReAct pattern
from langchain_classic.agents import create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Create a simplified ReAct prompt template
# agent_scratchpad will be auto-managed by create_react_agent
react_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the available tools to answer questions."),
    ("human", "{input}"),
])

agent = create_react_agent(
    llm=llm,
    tools=tool,
    prompt=react_prompt
)

agent

ValueError: Prompt missing required variables: {'tool_names', 'tools', 'agent_scratchpad'}

In [ ]:
# Agent is ready to use
agent_runnable = agent
agent_runnable

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMes

In [ ]:
#Execute agent directly (not through legacy AgentExecutor)
agent_executor = agent
agent_executor

AgentExecutor(verbose=True, agent=RunnableAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk'

In [ ]:
agent_executor.invoke({"input": "Tell me about langsmith"})



> Entering new AgentExecutor chain...


ValueError: variable agent_scratchpad should be a list of base messages, got  of type <class 'str'>